In [ ]:
```xml
<VSCode.Cell language="markdown">
# 💾 Comprehensive Caching Strategies Guide

**Multi-layer caching architectures for high-performance systems**

This guide covers:
- Caching layers and strategies
- Redis and Memcached
- Cache invalidation patterns
- Distributed caching
- Cache warming and preloading
- Cache monitoring and metrics
- Common caching mistakes
- Performance optimization with caching
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Caching Layers

```
Request Flow with Multi-Layer Caching

Client Request
    ↓
┌─────────────────────┐
│ Browser Cache (1hr) │ ← HTTP caching headers
└─────────────────────┘
    ↓ (miss)
┌─────────────────────┐
│  CDN Cache (1hr)    │ ← Geographic distribution
└─────────────────────┘
    ↓ (miss)
┌─────────────────────┐
│ Application Cache   │ ← Redis (5min)
│ (Query Results)     │
└─────────────────────┘
    ↓ (miss)
┌─────────────────────┐
│ Database Cache      │ ← DB query cache
│ (Buffer Pool)       │
└─────────────────────┘
    ↓ (miss)
┌─────────────────────┐
│ Disk I/O            │ ← Actual database
└─────────────────────┘
```

### Redis Cache Layer
```python
import redis
import json
from datetime import timedelta

class CacheManager:
    def __init__(self):
        self.redis = redis.Redis(
            host='redis',
            port=6379,
            decode_responses=True
        )
    
    def get_or_fetch(
        self,
        key: str,
        fetch_fn,
        ttl: int = 3600
    ):
        """Get from cache or fetch and cache"""
        
        # Try cache first
        cached = self.redis.get(key)
        if cached:
            logger.info(f"Cache hit: {key}")
            return json.loads(cached)
        
        logger.info(f"Cache miss: {key}")
        
        # Fetch fresh data
        data = fetch_fn()
        
        # Cache it
        self.redis.setex(
            key,
            ttl,
            json.dumps(data)
        )
        
        return data
    
    def invalidate(self, pattern: str):
        """Invalidate cache by pattern"""
        keys = self.redis.keys(pattern)
        if keys:
            self.redis.delete(*keys)
            logger.info(f"Invalidated {len(keys)} keys matching {pattern}")

# Usage
cache = CacheManager()

def get_user_profile(user_id: str):
    key = f"user:{user_id}:profile"
    return cache.get_or_fetch(
        key,
        lambda: db.query("SELECT * FROM users WHERE id = ?", user_id),
        ttl=3600  # 1 hour
    )
```

### Memcached for Distributed Caching
```python
from pymemcache.client.base import Client

memcached = Client(('memcached', 11211))

# Set cache with TTL
memcached.set(
    b'user:123:profile',
    json.dumps(user_data).encode(),
    expire=3600
)

# Get from cache
cached = memcached.get(b'user:123:profile')
if cached:
    user_data = json.loads(cached.decode())
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Cache Invalidation Patterns

### TTL (Time-To-Live)
```python
# Simple: expire after fixed time
redis.setex('key', 3600, value)  # Expires in 1 hour
```

### Event-Based Invalidation
```python
async def update_user(user_id: str, data: dict):
    """Update user and invalidate cache"""
    
    # Update in database
    await db.execute(
        "UPDATE users SET name = ? WHERE id = ?",
        (data['name'], user_id)
    )
    
    # Invalidate related caches
    cache.invalidate(f"user:{user_id}:*")
    cache.invalidate(f"profile:{user_id}:*")
    cache.invalidate(f"feed:*")  # Invalidate feeds that might show this user
    
    # Publish event
    await publish_event('user.updated', {'user_id': user_id})
```

### Cache-Aside Pattern
```python
async def get_data(key: str, source_fn):
    """Check cache first, fetch if missing"""
    
    cached = await redis.get(key)
    if cached:
        return json.loads(cached)
    
    # Cache miss: fetch from source
    data = await source_fn()
    
    # Store in cache
    await redis.setex(key, 3600, json.dumps(data))
    
    return data
```

### Write-Through Pattern
```python
async def save_data(key: str, data: dict):
    """Write to both cache and database"""
    
    # Write to database first (consistency)
    await db.save(key, data)
    
    # Then cache it
    await redis.setex(key, 3600, json.dumps(data))
```

### Write-Back (Write-Behind) Pattern
```python
async def save_data_async(key: str, data: dict):
    """Write to cache immediately, flush to DB periodically"""
    
    # Write to cache immediately (fast)
    await redis.setex(key, 3600, json.dumps(data))
    
    # Queue for database write
    await queue_for_persistence(key, data)

async def persist_from_cache():
    """Periodic job to write cache to database"""
    while True:
        items = await get_queued_items()
        for key, data in items:
            await db.save(key, data)
        await asyncio.sleep(60)  # Every 60 seconds
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Cache Warming

### Pre-load Cache on Startup
```python
async def warm_cache_on_startup():
    """Pre-load frequently accessed data"""
    
    logger.info("Warming cache...")
    
    # Load popular items
    popular_items = await db.query(
        "SELECT * FROM items ORDER BY views DESC LIMIT 1000"
    )
    
    for item in popular_items:
        key = f"item:{item['id']}"
        await redis.setex(key, 3600, json.dumps(item))
    
    logger.info(f"Warmed cache with {len(popular_items)} items")

app.on_startup(warm_cache_on_startup)
```

### Proactive Refresh
```python
async def refresh_expiring_cache():
    """Refresh cache before it expires"""
    
    # Find keys expiring soon
    keys = await redis.keys('*')
    
    for key in keys:
        ttl = await redis.ttl(key)
        
        # Refresh if < 5 minutes remaining
        if 0 < ttl < 300:
            data = await fetch_fresh_data(key)
            await redis.setex(key, 3600, json.dumps(data))
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Caching Pitfalls

| Mistake | Impact | Solution |
|---------|--------|----------|
| **Cache Stampede** | All requests hit DB when cache expires | Use locks or probabilistic TTL |
| **Stale Data** | Cached data is outdated | Implement proper invalidation |
| **Memory Leak** | Cache grows unbounded | Set max memory and eviction policy |
| **Cache Penetration** | Attacker requests non-existent keys | Cache negative results |
| **Large Objects** | Memory waste | Compress or split objects |

### Cache Stampede Solution
```python
import asyncio

class SmartCache:
    def __init__(self):
        self.redis = redis.Redis()
        self.locks = {}  # Track in-progress fetches
    
    async def get_or_fetch_safe(self, key: str, fetch_fn, ttl: int):
        """Prevent cache stampede"""
        
        # Check cache
        cached = await self.redis.get(key)
        if cached:
            return json.loads(cached)
        
        # Check if already being fetched
        if key in self.locks:
            # Wait for ongoing fetch
            return await self.locks[key]
        
        # Fetch with lock to prevent others
        lock = asyncio.Lock()
        self.locks[key] = lock
        
        try:
            data = await fetch_fn()
            await self.redis.setex(key, ttl, json.dumps(data))
            return data
        finally:
            del self.locks[key]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Caching Checklist

- [ ] Identify frequently accessed data
- [ ] Choose appropriate TTL
- [ ] Implement cache invalidation strategy
- [ ] Monitor hit/miss ratios
- [ ] Set up cache warming
- [ ] Handle cache failures gracefully
- [ ] Implement rate limiting
- [ ] Monitor memory usage
- [ ] Compress large objects
- [ ] Use consistent hashing for distributed cache

**Typical Cache Performance**: 100-1000x faster than database queries
</VSCode.Cell>
```